# Results summary: GNN-LSTM vs LSTM, multi-seed and graph ablation
Loads saved outputs from `processed/` (produced by `gnn_lstm.py`). Narrative in `progress_report.md`.

In [1]:
import json
import pandas as pd
from pathlib import Path

P = Path("processed")
stations = ["Bhaktapur", "Khumaltar", "Ratnapark", "Shankhapark"]
horizons = ["24", "48", "72"]

## 1. Regression: tuned LSTM vs GNN-LSTM (graph) vs shared model without graph (test MAE, 3 seeds)

In [2]:
# test MAE for 3 models x 3 seeds (42, 43, 44)
rows = []
for s in stations:
    for seed in (42, 43, 44):
        f = f"reg_{s}_tuned300_eval.json" if seed == 42 else f"reg_{s}_tuned300_seed{seed}_eval.json"
        r = json.load(open(P / f))
        for h in horizons:
            rows.append({"model": "lstm", "seed": seed, "station": s, "h": h, "mae": r[h]["mae"]})

for arm in ("graph", "nograph"):
    for seed in (42, 43, 44):
        r = json.load(open(P / f"ablate_reg_{arm}_seed{seed}.json"))["results"]
        for s in stations:
            for h in horizons:
                rows.append({"model": arm, "seed": seed, "station": s, "h": h, "mae": r[s][h]["mae"]})

d = pd.DataFrame(rows)
print(d.groupby(["station", "h", "model"]).mae.mean().unstack().round(2))
print()
print(d.groupby("model").mae.mean().round(2))

model           graph   lstm  nograph
station     h                        
Bhaktapur   24  16.30  14.37    14.80
            48  17.05  15.69    16.03
            72  17.39  16.45    16.45
Khumaltar   24  11.02  10.28    10.18
            48  12.12  11.41    11.23
            72  12.97  12.75    12.37
Ratnapark   24  15.00  15.40    13.91
            48  16.53  16.73    15.14
            72  18.39  17.94    16.78
Shankhapark 24  12.85  12.47    12.47
            48  13.80  13.44    13.34
            72  13.86  13.54    13.40

model
graph      14.77
lstm       14.21
nograph    13.84
Name: mae, dtype: float64


## 2. Regression ablation: no graph minus graph (paired by seed; negative = no-graph better)

In [3]:
# paired by seed: negative means no-graph is better
p = d.pivot_table(index=["station", "h", "seed"], columns="model", values="mae")
diff = (p["nograph"] - p["graph"]).groupby(["station", "h"]).agg(["mean", "std"]).round(2)
print(diff)
print("no-graph better in", (diff["mean"] < 0).sum(), "of", len(diff), "cells")

                mean   std
station     h             
Bhaktapur   24 -1.50  0.78
            48 -1.02  0.51
            72 -0.94  0.44
Khumaltar   24 -0.83  0.21
            48 -0.89  0.15
            72 -0.60  0.03
Ratnapark   24 -1.08  0.33
            48 -1.39  0.62
            72 -1.61  0.58
Shankhapark 24 -0.38  0.39
            48 -0.45  0.42
            72 -0.46  0.32
no-graph better in 12 of 12 cells


## 3. Classification ablation (test macro-F1, 3 seeds)

In [4]:
# classification ablation, macro-F1 and class 4 recall
c = pd.DataFrame(json.load(open(P / "ablate_clf_summary.json")))
print(c.groupby(["horizon", "arm"])[["macro_f1", "c4_recall"]].mean().round(3))

q = c.pivot_table(index=["station", "horizon", "seed"], columns="arm", values="macro_f1")
q["diff"] = q["graph"] - q["nograph"]
print()
print(q.groupby(["station", "horizon"])["diff"].agg(["mean", "std"]).round(3))

                 macro_f1  c4_recall
horizon arm                         
+24h    graph       0.723      1.000
        nograph     0.712      0.942
+48h    graph       0.665      1.000
        nograph     0.659      1.000
+72h    graph       0.603      1.000
        nograph     0.588      1.000

                      mean    std
station     horizon              
Bhaktapur   +24h     0.068  0.041
            +48h     0.080  0.025
            +72h     0.080  0.030
Khumaltar   +24h    -0.014  0.005
            +48h    -0.013  0.034
            +72h    -0.018  0.031
Ratnapark   +24h    -0.000  0.025
            +48h    -0.009  0.021
            +72h    -0.017  0.030
Shankhapark +24h    -0.012  0.015
            +48h    -0.032  0.065
            +72h     0.015  0.026


## 4. Classification: LSTM vs GNN-LSTM head-to-head

In [5]:
# lstm vs gnn classification, 3 seeds
print(pd.read_json(P / "clf_gnn_vs_lstm_summary.json"))

        station   h  f1_lstm  sd_lstm  c4_lstm  f1_gnn  sd_gnn  c4_gnn  n4  \
0     Bhaktapur  24    0.755    0.014        1   0.773   0.001       1  44   
1     Bhaktapur  48    0.681    0.030        1   0.704   0.025       1  44   
2     Bhaktapur  72    0.569    0.049        1   0.641   0.027       1  34   
3     Khumaltar  24    0.605    0.018        1   0.592   0.008       1  10   
4     Khumaltar  48    0.637    0.033        1   0.642   0.009       1  23   
5     Khumaltar  72    0.591    0.024        1   0.609   0.013       1  23   
6     Ratnapark  24    0.660    0.019        1   0.685   0.020       1  46   
7     Ratnapark  48    0.569    0.007        1   0.583   0.015       1  47   
8     Ratnapark  72    0.503    0.017        1   0.503   0.036       1  39   
9   Shankhapark  24    0.829    0.014        1   0.841   0.011       1  58   
10  Shankhapark  48    0.735    0.041        1   0.731   0.027       1  59   
11  Shankhapark  72    0.636    0.058        1   0.662   0.022  